# GRAS with a column that has no positive entry

GRAS extends RAS to matrices with negative entries. The example in {cite:t}`temurshoev-miller-bouwmeester-2013` is a small input-output table, updated to known row and column totals, whose net exports column has no positive entry. The iterative GRAS algorithm of {cite:t}`junius-oosterhaven-2003` divides by a weighted sum of the positive entries in each row and column, so the multiplier for that column is undefined. The paper derives the multipliers for rows and columns without a positive entry, and notes that the problem does not arise when GRAS is solved as an optimization problem, which is what `fit` does. This notebook reproduces the paper's Table 3.

In [1]:
import pandas as pd

import cge_modeling as cg

In [2]:
def cell_color(value):
    if value == 0.0:
        return "color: lightgray"
    if value < 0.0:
        return "color: firebrick"
    return ""


def style_table(frame, precision, caption):
    return (frame.style
            .format(precision=precision)
            .map(cell_color)
            .set_caption(caption)
            .set_table_styles([{"selector": "caption",
                                "props": "caption-side: top; font-weight: bold; text-align: left"}]))

## The data

The paper's Table 1 is the old table. It is Table 1 of {cite:t}`junius-oosterhaven-2003` with the two positive entries of the net exports column set to zero. GRAS updates its block of intermediate and final uses, three rows by four columns. Table 2 gives the new totals: total output for each row and total use for each column.

In [3]:
rows = ["Goods", "Services", "Net taxes"]
columns = ["Goods use", "Services use", "Consumption", "Net exports"]

old_block = pd.DataFrame(data=[[7.0, 3.0, 5.0, -3.0],
                               [2.0, 9.0, 8.0, 0.0],
                               [-2.0, 0.0, 2.0, 0.0]],
                         index=rows,
                         columns=columns)
row_totals = pd.Series(data=[15.0, 26.0, -1.0], index=rows)
column_totals = pd.Series(data=[9.0, 16.0, 17.0, -2.0], index=columns)

style_table(old_block,
            precision=0,
            caption="Old table (Table 1)")

,Goods use,Services use,Consumption,Net exports
Goods,7,3,5,-3
Services,2,9,8,0
Net taxes,-2,0,2,0


## Embedding the block in a SAM

GRAS adjusts a rectangular block so that each row sums to a known total and each column sums to a known total. `SymbolicSAM` works with a square SAM and imposes one rule on every account: its income, the row sum, equals its spending, the column sum. It has no direct notion of a row total for a block.

The block becomes a SAM by making each of its rows and columns an account and adding one extra account, `Totals`, that holds the known totals. `Totals` is a device for this embedding and does not appear in the paper.

- Each row account pays its known total to `Totals`. Goods pays 15, so the cell (`Totals`, Goods) is 15. That payment is the only spending of Goods, so the accounting identity forces the income of Goods, its row of the block, to sum to 15. This is the GRAS row constraint.
- Each column account receives its known total from `Totals`. Goods use receives 9, so the cell (Goods use, `Totals`) is 9. That receipt is the only income of Goods use, so the identity forces its spending, its column of the block, to sum to 9. This is the GRAS column constraint.
- `Totals` receives the row totals and pays out the column totals. Both sum to 40, so it balances too.

Constraints fix the cells of `Totals` at the known totals, so `fit` can only move the cells of the block. In the table below, the row and column of `Totals` are shaded.

In [4]:
accounts = [*rows, *columns, "Totals"]
prior = pd.DataFrame(0.0,
                     index=accounts,
                     columns=accounts)
prior.loc[rows, columns] = old_block
prior.loc["Totals", rows] = row_totals
prior.loc[columns, "Totals"] = column_totals

totals_account = {"background-color": "#e8eef6"}
(style_table(prior,
             precision=0,
             caption="Prior SAM with the Totals account")
 .set_properties(subset=pd.IndexSlice["Totals", :], **totals_account)
 .set_properties(subset=pd.IndexSlice[:, "Totals"], **totals_account))

,Goods,Services,Net taxes,Goods use,Services use,Consumption,Net exports,Totals
Goods,0,0,0,7,3,5,-3,0
Services,0,0,0,2,9,8,0,0
Net taxes,0,0,0,-2,0,2,0,0
Goods use,0,0,0,0,0,0,0,9
Services use,0,0,0,0,0,0,0,16
Consumption,0,0,0,0,0,0,0,17
Net exports,0,0,0,0,0,0,0,-2
Totals,15,26,-1,0,0,0,0,0


## Fitting

{cite:t}`lenzen-wood-gallego-2007` corrected the GRAS objective of Junius and Oosterhaven, and equation (1) of the paper uses the corrected form. The objective of `fit` is zero on the cells of `Totals`, because the constraints hold them at their prior values. On the block cells it equals equation (1) plus the total magnitude of the prior, a constant, so the two objectives have the same minimizer. Table 3 is published to three decimals, so the call tightens the CLARABEL tolerances to make the solver error negligible at that precision.

In [5]:
sam = cg.SymbolicSAM(prior)
sam.subject_to(row_totals=sam.sel(row="Totals", col=rows) == row_totals,
               column_totals=sam.sel(row=columns, col="Totals") == column_totals)
estimate = sam.fit(solver="CLARABEL",
                   tol_gap_abs=1e-12,
                   tol_gap_rel=1e-12,
                   tol_feas=1e-12)

new_block = estimate.sam.loc[rows, columns]
style_table(new_block,
            precision=3,
            caption="New table from fit")

,Goods use,Services use,Consumption,Net exports
Goods,8.424,3.375,5.200,-2.000
Services,3.001,12.625,10.374,0.000
Net taxes,-2.425,0.000,1.425,0.000


In [6]:
published = pd.DataFrame(data=[[8.424, 3.375, 5.200, -2.000],
                               [3.001, 12.625, 10.374, 0.0],
                               [-2.425, 0.0, 1.425, 0.0]],
                         index=rows,
                         columns=columns)

(style_table((new_block - published).abs(),
             precision=4,
             caption="Absolute difference from Table 3, shaded up to the rounding bound 0.0005")
 .background_gradient(cmap="Reds",
                      vmin=0.0,
                      vmax=0.0005))

,Goods use,Services use,Consumption,Net exports
Goods,0.0003,0.0002,0.0004,0.0000
Services,0.0000,0.0002,0.0002,0.0000
Net taxes,0.0003,0.0000,0.0003,0.0000


Every cell matches Table 3 to within 0.0004, inside the rounding of the published three decimals.

## References

```{bibliography}
```

## Authors

- Created by Jesse Grabowski, November 22, 2026

## Watermark

In [7]:
%load_ext watermark
%watermark -n -u -v -iv -m -w

Last updated: Sun, 22 Nov 2026

Python implementation: CPython
Python version       : 3.12.13
IPython version      : 9.18.1

Compiler    : Clang 19.1.7 
OS          : Darwin
Release     : 25.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

cge_modeling: 0.0.7.dev15+g93cb61e4e.d20261120
pandas      : 3.1.1

Watermark: 2.6.0

